<a href="https://colab.research.google.com/github/FastPickle56/fungal-temp-analysis/blob/main/fungal_species_puller_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fungal Reference Genome Species Puller (NCBI)

This notebook generates a reproducible list of fungal species and their **NCBI RefSeq reference genome** assemblies (green-check).

**Outputs**
- `fungi_reference_assemblies_clean.csv`: species → best assembly (after removing ambiguous placeholders like `sp.` / `uncultured`)
- `fungi_reference_assemblies_reference_only.csv`: **strict** RefSeq *reference genome* rows only (green-check only)
- `fungi_species_reference_only.txt`: species list for downstream literature querying

**Method**
1. Query NCBI Datasets API for all assemblies under taxon **Fungi (taxid 4751)**.
2. Pick the best assembly per species (prioritizing RefSeq + reference/representative when present).
3. Remove placeholder species labels (e.g., `sp.`, `uncultured`).
4. Restrict to `refseq_category == "reference genome"` (green-check only).


In [1]:
# ================================
# BUILD SPECIES LIST: ALL FUNGI WITH GENOMES
# From NCBI Datasets API (taxid 4751)
# ================================

import requests
import pandas as pd
from datetime import datetime
from typing import Dict, Any, List, Optional, Tuple
import time

BASE = "https://api.ncbi.nlm.nih.gov/datasets/v2"
FUNGI_TAXID = 4751  # Fungi

# ----------------------------
# Helper functions
# ----------------------------

def _safe_get(d: Dict[str, Any], path: List[str], default=None):
    cur = d
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return default
        cur = cur[p]
    return cur

def _parse_date(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%Y-%m-%dT%H:%M:%S.%f", "%Y-%m-%dT%H:%M:%SZ"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    return None

def normalize_species_name(name: str) -> str:
    """
    Reduce organism names to binomial form (Genus species).
    Removes strain info.
    """
    if not name:
        return ""
    parts = name.split()
    if len(parts) >= 2:
        return f"{parts[0]} {parts[1]}"
    return name

# ----------------------------
# Pull all fungal assemblies
# ----------------------------

def fetch_fungi_assembly_reports(page_size: int = 1000, delay_s: float = 0.12):
    url = f"{BASE}/genome/taxon/{FUNGI_TAXID}/dataset_report"
    params = {"page_size": str(page_size)}

    reports = []
    page_token = None

    while True:
        if page_token:
            params["page_token"] = page_token
        else:
            params.pop("page_token", None)

        r = requests.get(url, params=params, timeout=120)
        r.raise_for_status()
        data = r.json()

        batch = data.get("reports", [])
        reports.extend(batch)

        page_token = data.get("next_page_token")
        time.sleep(delay_s)

        if not page_token:
            break

    return reports

# ----------------------------
# Rank assemblies per species
# ----------------------------

def score_assembly(rep: Dict[str, Any]) -> int:
    accession = rep.get("accession", "") or ""
    source_db = (rep.get("source_database") or "").upper()
    asm = rep.get("assembly_info") or {}

    refseq_cat = (asm.get("refseq_category") or "").lower()
    is_reference_like = ("reference" in refseq_cat) or ("representative" in refseq_cat)

    is_refseq = accession.startswith("GCF_") or ("REFSEQ" in source_db)

    d = _parse_date(asm.get("release_date"))

    score = 0
    score += 1000 if is_refseq else 0
    score += 600 if is_reference_like else 0
    if d:
        score += int(d.timestamp() // 86400)

    return score

def build_species_reference_table(reports: List[Dict[str, Any]]) -> pd.DataFrame:
    best_by_species: Dict[str, Tuple[int, Dict[str, Any]]] = {}

    for rep in reports:
        org_name = _safe_get(rep, ["organism", "organism_name"], "") or ""
        species = normalize_species_name(org_name)

        if not species:
            continue

        accession = rep.get("accession")
        if not accession:
            continue

        s = score_assembly(rep)

        if (species not in best_by_species) or (s > best_by_species[species][0]):
            best_by_species[species] = (s, rep)

    rows = []

    for species, (s, rep) in best_by_species.items():
        asm = rep.get("assembly_info") or {}

        rows.append({
            "species": species,
            "organism_name_full": _safe_get(rep, ["organism", "organism_name"], None),
            "tax_id": _safe_get(rep, ["organism", "tax_id"], None),
            "strain": _safe_get(rep, ["organism", "infraspecific_names", "strain"], None),

            "assembly_accession": rep.get("accession"),
            "paired_accession": rep.get("paired_accession"),
            "source_database": rep.get("source_database"),

            "assembly_name": asm.get("assembly_name"),
            "assembly_level": asm.get("assembly_level"),
            "assembly_status": asm.get("assembly_status"),
            "refseq_category": asm.get("refseq_category"),
            "release_date": asm.get("release_date"),
            "bioproject_accession": asm.get("bioproject_accession"),
            "biosample_accession": _safe_get(asm, ["biosample", "accession"], None),

            "scoring_rank": s,
        })

    df = pd.DataFrame(rows)

    return df.sort_values("species").reset_index(drop=True)

# ================================
# RUN EVERYTHING
# ================================

print("Pulling all fungal assemblies from NCBI...")
reports = fetch_fungi_assembly_reports()

print("Total assembly records pulled:", len(reports))

df = build_species_reference_table(reports)

print("Unique fungal species with at least one genome:", len(df))
display(df.head(10))

# Save files
df.to_csv("fungi_reference_assemblies.csv", index=False)

with open("fungi_species.txt", "w", encoding="utf-8") as f:
    for sp in df["species"].astype(str):
        f.write(sp + "\n")

print("\nFiles saved:")
print(" - fungi_reference_assemblies.csv")
print(" - fungi_species.txt")

Pulling all fungal assemblies from NCBI...
Total assembly records pulled: 24314
Unique fungal species with at least one genome: 5846


,species,organism_name_full,tax_id,strain,assembly_accession,paired_accession,source_database,assembly_name,assembly_level,assembly_status,refseq_category,release_date,bioproject_accession,biosample_accession,scoring_rank
0,Aaosphaeria arxii,Aaosphaeria arxii CBS 175.79,1450172,CBS 175.79,GCF_010015735.1,GCA_010015735.1,SOURCE_DATABASE_REFSEQ,Aaoar1,Scaffold,current,reference genome,2020-01-29,PRJNA234399,SAMN05661102,19890
1,Aaosphaeria pasadenensis,Aaosphaeria pasadenensis,3025424,FJI-L9-BK-P1,GCA_022813625.1,None,SOURCE_DATABASE_GENBANK,ASM2281362v1,Scaffold,current,reference genome,2022-04-04,PRJNA800051,SAMN25226780,19686
2,Abortiporus biennis,Abortiporus biennis,137743,LGAM 436,GCA_047301825.1,None,SOURCE_DATABASE_GENBANK,ASM4730182v1,Contig,current,None,2025-01-29,PRJNA1117344,SAMN41564068,20117
3,Absidia cylindrospora,Absidia cylindrospora,213030,None,GCA_977092135.1,None,SOURCE_DATABASE_GENBANK,gzAbsCyli1,Contig,current,reference genome,2025-12-07,PRJEB95971,SAMEA118933587,21029
4,Absidia glauca,Absidia glauca,4829,CBS 101.48 substr. RVII-324 met-,GCA_900079185.1,None,SOURCE_DATABASE_GENBANK,AG_v1,Scaffold,current,reference genome,2016-04-15,PRJEB13349,SAMEA3923633,17506
5,Absidia healeyae,Absidia healeyae,2733465,UoMAU1,GCA_019677225.1,None,SOURCE_DATABASE_GENBANK,ASM1967722v1,Scaffold,current,reference genome,2021-08-18,PRJNA630489,SAMN14838375,19457
6,Absidia repens,Absidia repens,90262,NRRL 1336,GCA_002105175.1,None,SOURCE_DATABASE_GENBANK,Absrep1,Contig,current,reference genome,2017-04-20,PRJNA330702,SAMN05421884,17876
7,Absidia sp.,Absidia sp. YW-2025a,3423207,CGMCC 3.27810,GCA_050656845.1,None,SOURCE_DATABASE_GENBANK,ASM5065684v1,Scaffold,current,None,2025-06-02,PRJNA1262716,SAMN48480011,20241
8,Acanthothecis fontana,Acanthothecis fontana,2569847,None,GCA_964256675.1,None,SOURCE_DATABASE_GENBANK,public_SRR14721923_metabat2_bin.2,Scaffold,current,reference genome,2024-09-21,PRJEB77567,SAMEA115958113,20587
9,Acaromyces ingoldii,Acaromyces ingoldii,215250,MCA 4198,GCF_003144295.1,GCA_003144295.1,SOURCE_DATABASE_REFSEQ,Acain1,Scaffold,current,reference genome,2018-05-21,PRJNA247834,SAMN02787018,19272



Files saved:
 - fungi_reference_assemblies.csv
 - fungi_species.txt


In [2]:
# ================================
# CLEAN SPECIES LIST (remove placeholders like "sp.", "aff.", etc.)
# ================================

import pandas as pd

# Load the file created in previous step
df = pd.read_csv("fungi_reference_assemblies.csv")

# Terms that indicate ambiguous / non-species entries
bad_tokens = [
    " sp.",
    " aff.",
    " cf.",
    "uncultured",
    "unidentified",
    "environmental",
    "metagenome"
]

# Build regex-safe search pattern
pattern = "|".join([t.replace(".", r"\.") for t in bad_tokens])

# Keep only real species
mask = ~df["species"].str.contains(pattern, case=False, na=False)

df_clean = df[mask].copy()

# Save clean versions
df_clean.to_csv("fungi_reference_assemblies_clean.csv", index=False)

with open("fungi_species_clean.txt", "w") as f:
    for s in df_clean["species"].tolist():
        f.write(s + "\n")

print("Original species count:", len(df))
print("Clean species count:", len(df_clean))

Original species count: 5846
Clean species count: 5337


In [3]:
import pandas as pd

# Load the cleaned dataset
df = pd.read_csv("fungi_reference_assemblies_clean.csv")

# Strict filter: ONLY green-check equivalent
df_ref_only = df[df["refseq_category"] == "reference genome"].copy()

print("Total cleaned species:", len(df))
print("Strict reference genomes:", len(df_ref_only))

# Save strict versions
df_ref_only.to_csv("fungi_reference_assemblies_reference_only.csv", index=False)

with open("fungi_species_reference_only.txt", "w") as f:
    for s in df_ref_only["species"].tolist():
        f.write(s + "\n")

print("\nSaved:")
print(" - fungi_reference_assemblies_reference_only.csv")
print(" - fungi_species_reference_only.txt")

Total cleaned species: 5337
Strict reference genomes: 4707

Saved:
 - fungi_reference_assemblies_reference_only.csv
 - fungi_species_reference_only.txt


In [4]:
from google.colab import files

files.download("fungi_reference_assemblies_reference_only.csv")
files.download("fungi_species_reference_only.txt")

# (Optional) If downloads fail, run the zip cell below.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# If Colab download fails, zip outputs and download the zip instead
!zip -r fungi_reference_only_outputs.zip fungi_reference_assemblies_reference_only.csv fungi_species_reference_only.txt
from google.colab import files
files.download("fungi_reference_only_outputs.zip")


## Notes

- If you hit Colab `Failed to fetch` on download, zip the outputs and download the zip instead.
- Start downstream text-mining on a sampled subset first (e.g., 200 species) before scaling to all species.
